# Algorithm 1 trên ImageNet-R · Reimplement của nhóm

Notebook 06 dùng chung `run_algorithm1`, vòng lặp Phase 1, cache embedding, bộ phân loại ridge và cách xuất metric với các notebook 01–05. Thí nghiệm này dùng SSF được cài đặt trên ViT-B/16 của timm, cùng checkpoint AugReg và cùng split **ImageNet-R train/test** với preset 7 của tác giả. Seed thứ tự lớp là 1993, gồm 10 task × 20 lớp, batch hiệu dụng 48, 20 epoch Phase 1 và (M=10000). [Preset gốc](https://github.com/McDonnell-Research-Lab/RanPAC/blob/cf4b301d18b0c27db030f4371b72b768005ae58a/args/imagenetr_publish.csv#L9)

1. Đưa **RanPAC-reimplement-imagenetr.zip** vào **MyDrive/RanPAC_implement**, cùng thư mục mã nguồn của notebook 04. Mở notebook trong Colab, chọn **Runtime → Change runtime type → GPU** và chạy các cell từ trên xuống.
2. Nếu chưa có dữ liệu, notebook tải split ImageNet-R từ README tác giả và lưu tại `MyDrive/imagenet-r.zip` để cả hai notebook dùng chung. Nếu đã có ZIP, TAR/TAR.GZ hoặc thư mục có `train/` và `test/` ở đường dẫn khác, đặt `DATASET_INPUT` theo đường dẫn đó. [Nguồn split](https://github.com/McDonnell-Research-Lab/RanPAC/blob/cf4b301d18b0c27db030f4371b72b768005ae58a/README.md#datasets-tested-on-in-mcdonnell-et-al)
3. Để thử trước, đặt `STOP_AFTER_EPOCH = 1` và giữ nguyên `PHASE1_EPOCHS = 20`. Để tiếp tục, đặt `RESUME = True`, giữ nguyên tên run và các tham số thí nghiệm, rồi nâng hoặc bỏ giới hạn dừng. Dùng `RUN_LABEL` mới nếu muốn chạy lại từ đầu. Nếu thiếu VRAM, giảm `MICROBATCH` và bắt đầu run mới.
4. Cell cuối đọc CSV source gốc và CSV reimplement để xuất chênh lệch từng task. Có thể chỉ định đường dẫn CSV gốc nếu đã chuyển file sang nơi khác.

Cả hai bên dùng cùng phương pháp SSF và cách lấy trung bình accuracy theo các nhóm lớp đã thấy. Bản nhóm giữ cách sinh ma trận chiếu, chia 80/20, giải ridge và seed huấn luyện của runner reimplement. Các khác biệt đó có thể làm kết quả lệch giữa hai bản.

## 1. Cài thư viện và mở mã nguồn

Gói ZIP chứa các module dùng chung. Notebook giải nén gói vào ổ đĩa Colab. Tạo secret `WANDB_API_KEY` trong Colab và bật quyền truy cập cho notebook này để log vào `pnp123/ML` như notebook 04.

In [ ]:
%pip install -q timm==1.0.22 wandb==0.31.0 gdown==5.2.0

In [ ]:
from pathlib import Path
import sys
import zipfile

import torch
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive")
SOURCE_DIR = DRIVE / "RanPAC_implement"
SOURCE_ZIP = SOURCE_DIR / "RanPAC-reimplement-imagenetr.zip"
EXTRACTED_SOURCE_DIR = Path("/content/RanPAC-reimplement-imagenetr")
WORK_DIR = DRIVE / "ranpac-runs"
DEFAULT_DATASET_INPUT = DRIVE / "imagenet-r.zip"
DATASET_INPUT = DEFAULT_DATASET_INPUT  # Hoặc dùng đường dẫn tới split đã có.

with zipfile.ZipFile(SOURCE_ZIP) as archive:
    archive.extractall("/content")
sys.path.insert(0, str(EXTRACTED_SOURCE_DIR / 'src'))

from ranpac_data import atomic_output
from ranpac_full import run_algorithm1
from ranpac_imagenetr import ImageNetR

if not torch.cuda.is_available():
    raise RuntimeError("Chọn GPU runtime trước khi chạy huấn luyện.")
print("GPU:", torch.cuda.get_device_name(0))

## 2. Chuẩn bị cùng split với source gốc

Cell tải tự động khi thiếu tệp ở đường dẫn mặc định. Nếu Google Drive báo giới hạn tải, có thể dùng liên kết OneDrive trong README tác giả rồi đặt `DATASET_INPUT` theo tệp đã tải. Nếu còn cùng runtime với notebook source gốc, có thể trỏ `DATASET_INPUT` thẳng tới `/content/RanPAC-original-imagenetr/data/imagenet-r`. Cell kiểm tra lớp và in số ảnh của cả hai split.

In [ ]:
import shutil
import tarfile
from contextlib import nullcontext
from tempfile import TemporaryDirectory

import gdown

IMAGE_NET_R_DRIVE_ID = "1SG4TbiL8_DooekztyCVK8mPmfhMo8fkR"
DATASET_ROOT = Path("/content/ranpac-data/imagenet-r")
data = None
if (DATASET_INPUT.is_relative_to("/content/RanPAC-original-imagenetr")
        and (DATASET_INPUT / "train").is_dir() and (DATASET_INPUT / "test").is_dir()):
    DATASET_ROOT = DATASET_INPUT
elif not DATASET_ROOT.is_dir():
    download = not DATASET_INPUT.exists()
    if download and DATASET_INPUT != DEFAULT_DATASET_INPUT:
        raise FileNotFoundError(f"Sửa DATASET_INPUT theo đường dẫn thật: {DATASET_INPUT}")
    DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)
    input_context = atomic_output(DATASET_INPUT) if download else nullcontext(DATASET_INPUT)
    with input_context as input_path, TemporaryDirectory(dir="/content") as temp:
        if download:
            print(f"Đang tải split ImageNet-R vào {DATASET_INPUT}")
            result = gdown.download(id=IMAGE_NET_R_DRIVE_ID, output=str(input_path))
            if result is None:
                raise RuntimeError("Tải ImageNet-R thất bại. Xem liên kết tải trong README tác giả.")

        stage = Path(temp)
        if input_path.is_dir():
            shutil.copytree(input_path, stage / "dataset")
        elif zipfile.is_zipfile(input_path):
            with zipfile.ZipFile(input_path) as archive:
                archive.extractall(stage)
        elif tarfile.is_tarfile(input_path):
            with tarfile.open(input_path) as archive:
                archive.extractall(stage, filter="data")
        else:
            raise ValueError("Cần một tệp ZIP, TAR/TAR.GZ hoặc thư mục dataset.")

        roots = [p.parent for p in stage.rglob("train")
                 if p.is_dir() and (p.parent / "test").is_dir()]
        if len(roots) != 1:
            raise ValueError("Cần đúng một thư mục chứa cả train/ và test/. Kiểm tra lại bản split.")
        data = ImageNetR(roots[0])
        roots[0].rename(DATASET_ROOT)
else:
    print("Đang dùng split đã giải nén trong runtime. Nếu đổi split, hãy dùng runtime mới.")

if data is None:
    data = ImageNetR(DATASET_ROOT)
for split, images in data.splits.items():
    print(f"{split}: {len(images):,} ảnh, {len(images.classes)} lớp")
print("Mã đối chiếu danh sách ảnh:", data.manifest_sha256)
del data

## 3. Chạy Algorithm 1 của nhóm

Run lưu ở `MyDrive/ranpac-runs/runs/full-imagenetr-s1993-<RUN_LABEL>/`. Có checkpoint sau mỗi epoch Phase 1 và mỗi task Phase 2. Mặc định log online như notebook 04. Nếu đã tạo checkpoint với chế độ W&B khác, giữ nguyên chế độ đó khi resume. Để so sánh cấu hình mặc định, chạy đủ tới task 10 với FP32.

In [ ]:
RUN_LABEL = "main"
RESUME = False
PHASE1_EPOCHS = 20
STOP_AFTER_EPOCH = None
STOP_AFTER_TASK = 10
MICROBATCH = 48

WANDB_MODE = "online"  # Có thể chọn "offline" hoặc "disabled" trước khi chạy mới.
WANDB_ENTITY = "pnp123"
WANDB_PROJECT = "ML"

result = run_algorithm1(
    WORK_DIR, imagenetr_root=DATASET_ROOT, run_label=RUN_LABEL, resume=RESUME,
    phase1_epochs=PHASE1_EPOCHS, stop_after_epoch=STOP_AFTER_EPOCH,
    stop_after_task=STOP_AFTER_TASK, microbatch=MICROBATCH, precision="fp32",
    temporary_head="cosine", projection_dim=10000, fixed_lambda=None,
    feature_batch_size=48, stats_device="cpu", workers=2,
    entity=WANDB_ENTITY, project=WANDB_PROJECT, wandb_mode=WANDB_MODE,
)

## 4. So sánh với source gốc

Cell dùng `ave_acc` từ file `imagenetr_publish_7.csv` của source gốc và `avg_accuracy_pct` từ `metrics.csv` của bản nhóm. Mặc định nó chọn lượt source gốc hoàn tất gần nhất trong thư mục notebook trước. Nếu có nhiều lượt và muốn dùng một mốc cụ thể, điền `ORIGINAL_CSV`. Chênh lệch tính theo **reimplement trừ source gốc**, đơn vị điểm phần trăm. Có thể chạy lại riêng cell này sau khi đã có cả hai CSV.

In [ ]:
import csv
import math
import matplotlib.pyplot as plt

ORIGINAL_CSV = None  # Hoặc điền đường dẫn tuyệt đối tới imagenetr_publish_7.csv.
GROUP_RUN_LABEL = None  # Hoặc điền tên run khác đã hoàn tất ở bước 3.
group_run_label = GROUP_RUN_LABEL or globals().get("RUN_LABEL", "main")
group_dir = WORK_DIR / "runs" / f"full-imagenetr-s1993-{group_run_label}"

if ORIGINAL_CSV is None:
    original_root = DRIVE / "RanPAC-original-imagenetr-results"
    candidates = []
    for folder in sorted(original_root.glob("preset-7-*")):
        exit_file = folder / "exit_code.txt"
        path = folder / "results" / "imagenetr_publish_7.csv"
        if path.is_file() and exit_file.is_file() and exit_file.read_text().strip() == "0":
            candidates.append(path)
    if not candidates:
        raise FileNotFoundError("Chưa tìm thấy run source gốc. Điền ORIGINAL_CSV.")
    original_csv = candidates[-1]
else:
    original_csv = Path(ORIGINAL_CSV)

def read_metrics(path):
    with path.open(newline="", encoding="utf-8") as file:
        return list(csv.DictReader(file))

original = read_metrics(original_csv)
group = read_metrics(group_dir / "metrics.csv")
tasks = list(range(1, 11))
if len(original) != len(tasks) or [int(row["task"]) for row in group] != tasks:
    raise ValueError("Hai run phải hoàn tất đủ 10 task theo thứ tự.")

comparison = []
for task, authors, reimplementation in zip(tasks, original, group):
    a = float(authors["ave_acc"])
    b = float(reimplementation["avg_accuracy_pct"])
    if any(not math.isfinite(value) or not 0 <= value <= 100 for value in (a, b)):
        raise ValueError(f"Accuracy không hợp lệ ở task {task}")
    comparison.append((task, a, b, round(b - a, 4)))

with (group_dir / "comparison.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["task", "original_a_t_pct", "reimplement_a_t_pct", "delta_pp"])
    writer.writerows(comparison)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tasks, [row[1] for row in comparison], marker="o", label="Source gốc")
ax.plot(tasks, [row[2] for row in comparison], marker="s", label="Reimplement")
ax.set(xlabel="Task", ylabel="A[t] (%)", xticks=tasks)
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(group_dir / "comparison.png", dpi=160)
plt.show()

_, a10, b10, delta = comparison[-1]
print("Source gốc:", original_csv)
print("Reimplement:", group_dir / "metrics.csv")
print(f"A[10]: gốc {a10:.2f}% | reimplement {b10:.2f}% | chênh lệch {delta:+.2f} điểm %")
print("Đã ghi comparison.csv và comparison.png vào", group_dir)